# 02 — Clustering Smart Meters

Extract daily shape features per meter, run K-Means (k=2–3), visualise centroids, save `data/clusters.csv`.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

from src.data.loader import load_raw

plt.rcParams['figure.dpi'] = 110
sns.set_theme(style='whitegrid')

STEPS_PER_DAY = 96
N_CLUSTERS = 3   # <-- change and re-run to explore

## 1. Load data

In [ ]:
df = load_raw('../data/power.pk')
T, N = df.shape
print(f'Loaded: {T} timesteps × {N} meters')

## 2. Daily shape features

In [ ]:
n_days = T // STEPS_PER_DAY
arr = df.values[:n_days * STEPS_PER_DAY, :].reshape(n_days, STEPS_PER_DAY, N)  # (days, 96, N)

def per_meter_features(arr):
    """arr: (days, 96, N)  ->  features: (N, n_feat)"""
    arr_T = arr.transpose(2, 0, 1)          # (N, days, 96)
    
    mean_cons = np.nanmean(arr_T, axis=(1, 2))   # (N,)
    std_cons  = np.nanstd( arr_T, axis=(1, 2))   # (N,)
    
    # Peak hour (0-23) — average per meter
    daily_means = np.nanmean(arr_T, axis=1)      # (N, 96)
    peak_step   = np.argmax(daily_means, axis=1) # (N,)
    peak_hour   = peak_step / 4.0
    
    # Morning/evening ratio (06-10 vs 18-22)
    morning = np.nanmean(daily_means[:, 24:40], axis=1)   # 06-10h
    evening = np.nanmean(daily_means[:, 72:88], axis=1)   # 18-22h
    me_ratio = (morning + 1e-8) / (evening + 1e-8)
    
    # Variability (std / mean)
    variability = std_cons / (mean_cons + 1e-8)
    
    return np.column_stack([mean_cons, std_cons, peak_hour, me_ratio, variability])

features = per_meter_features(arr)
print('Feature matrix shape:', features.shape)
feat_names = ['mean', 'std', 'peak_hour', 'morning_evening_ratio', 'variability']
pd.DataFrame(features, columns=feat_names).describe()

## 3. Elbow + Silhouette

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(np.nan_to_num(features))

inertias, sil_scores = [], []
K_range = range(2, 8)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X, labels))
    print(f'  k={k}  inertia={km.inertia_:.1f}  silhouette={sil_scores[-1]:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(K_range), inertias, 'o-', color='steelblue')
axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertia'); axes[0].set_title('Elbow')
axes[1].plot(list(K_range), sil_scores, 's-', color='coral')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette score'); axes[1].set_title('Silhouette')
plt.tight_layout(); plt.show()

## 4. Final clustering

In [ ]:
km_final = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=15)
cluster_labels = km_final.fit_predict(X)

print('Cluster sizes:', {c: int((cluster_labels == c).sum()) for c in range(N_CLUSTERS)})

## 5. Centroid profiles

In [ ]:
hours = np.arange(STEPS_PER_DAY) / 4
daily_profiles = np.nanmean(arr, axis=0).T   # (N, 96) — per-meter average daily profile

fig, axes = plt.subplots(1, N_CLUSTERS, figsize=(5 * N_CLUSTERS, 4), sharey=True)
colors = ['steelblue', 'coral', 'mediumseagreen', 'orchid'][:N_CLUSTERS]

for cid in range(N_CLUSTERS):
    mask = cluster_labels == cid
    profiles_c = daily_profiles[mask]    # (n_c, 96)
    mu = profiles_c.mean(0)
    sigma = profiles_c.std(0)
    axes[cid].fill_between(hours, mu - sigma, mu + sigma, alpha=0.3, color=colors[cid])
    axes[cid].plot(hours, mu, color=colors[cid], linewidth=2)
    axes[cid].set_title(f'Cluster {cid}  (n={mask.sum()})')
    axes[cid].set_xlabel('Hour')
    if cid == 0: axes[cid].set_ylabel('Consumption')

plt.suptitle('Mean ± std daily profile per cluster', fontsize=12)
plt.tight_layout(); plt.show()

## 6. Feature scatter

In [ ]:
feat_df = pd.DataFrame(features, columns=feat_names)
feat_df['cluster'] = cluster_labels

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors_arr = [colors[c] for c in cluster_labels]

axes[0].scatter(feat_df['mean'], feat_df['peak_hour'], c=colors_arr, alpha=0.6, s=20)
axes[0].set_xlabel('Mean consumption'); axes[0].set_ylabel('Peak hour'); axes[0].set_title('Mean vs Peak hour')

axes[1].scatter(feat_df['variability'], feat_df['morning_evening_ratio'], c=colors_arr, alpha=0.6, s=20)
axes[1].set_xlabel('Variability'); axes[1].set_ylabel('Morning/Evening ratio'); axes[1].set_title('Variability vs M/E ratio')

plt.tight_layout(); plt.show()

## 7. Save cluster labels

In [ ]:
clusters_df = pd.DataFrame({
    'meter_idx': np.arange(N),
    'cluster_id': cluster_labels,
})
if hasattr(df.columns, 'tolist'):
    clusters_df['meter_id'] = df.columns.tolist()

clusters_df.to_csv('../data/clusters.csv', index=False)
print('Saved → ../data/clusters.csv')
clusters_df.groupby('cluster_id').size()